# 02b — Diagnostic: pipeline vs Ultralytics `model.val()` gap

**Purpose.** Notebook 02 found that our pipeline and Ultralytics `model.val()` differ by 0.29 pts
in mAP@0.5:0.95 and by **1.10 pts in mAP@0.5** on the same 200 calib frames. This notebook works
out whether the gap comes from **our predictions** or from **our metric implementation**.
It runs PyTorch FP32 YOLO11s with identical settings (conf 0.001, NMS IoU 0.7, max_det 300,
640 px square letterbox, batch 1) on the **same seeded 200-frame calib mini-set** and computes:
- **A**: our predictions (`YOLO.predict`), scored by our evaluator (pycocotools).
- **B**: Ultralytics' own `model.val()` predictions (`save_json`), scored by our evaluator.
- **C**: the same `model.val()` predictions, scored by Ultralytics' metric.

**B − A** is the prediction part and **C − B** the metric part, reported for both mAP@0.5:0.95
and mAP@0.5. Agreement now requires **both** metrics within 0.5 pts. If the end-to-end gap comes
from the metric implementation, the 0.5 pt rule applies to |C − B| (identical predictions)
instead. **Calib frames only; the test split is not touched.** No TensorRT engines are needed.

**Before running**
1. *Add Data* → **`tcjpaperdatasets`**.
2. *Settings* → **Accelerator: GPU T4 x2** (one GPU pinned), **Internet: On**.
3. *Add-ons → Secrets* → tick **`GH_TOKEN`** for **this** notebook (Secrets attach per notebook).
4. Run with `SMOKE_TEST = True` (50 frames, ≈5 min including installs). If it passes, set
   `SMOKE_TEST = False` → *Save Version → Save & Run All* (200 frames, ≈6–8 min).

**Download afterwards:** `results_02b_val_diagnostic.zip` → unzip into the repo root
(`results/sanity/val_diag_02b_yolo11s/diag.json`, `logs/`). Don't download the notebook's
working folder: it contains BDD frames.


In [ ]:
# ===================== CELL 1: CONFIG + ENVIRONMENT CHECK =====================
# Edit nothing below for a normal run except SMOKE_TEST.
SMOKE_TEST = True   # True: tiny subset (run this first). False: full run.
NOTEBOOK_REV = "3eaf887379c4"   # hash of this notebook's generated source; checked against the cloned repo

CONFIG = {
    "notebook": "02b_val_diagnostic",
    "repo_https": "https://github.com/vinaykamble-nsio/adapres.git",       # private repo, cloned with Secret GH_TOKEN
    "repo_ref": "main",
    "repo_dir": "/tmp/adapres",                     # /tmp: not saved into notebook output
    "repo_zip_marker": "adapres/GIT_COMMIT",        # fallback: repo zip attached as a dataset
    "config_path": "configs/default.yaml",          # relative to repo_dir
    "overrides": {                                  # applied on top of default.yaml
        "paths": {"input_root": "/kaggle/input", "work_root": "/kaggle/working"},
    },
    "cuda_visible_devices": "0",                    # T4 x2 -> pin to one GPU
    "required_gpu_substring": "T4",
    "model_override": None,   # e.g. {"name": "yolo11m", "weights": "yolo11m.pt"} if the size rule triggers
}

import os, sys, subprocess, json, platform, base64, glob, shutil, time
os.environ["CUDA_VISIBLE_DEVICES"] = CONFIG["cuda_visible_devices"]  # before any CUDA import
T0 = time.time()

def sh(cmd, check=True):
    """Run a shell command, print its output, raise with the real error on failure."""
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True)
    print(r.stdout[-6000:])
    if r.stderr.strip():
        print(r.stderr[-6000:], file=sys.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r

sh("nvidia-smi")
print("python", sys.version)
for mod in ("torch", "onnxruntime", "tensorrt", "ultralytics"):
    try:
        m = __import__(mod)
        print(f"{mod} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{mod} NOT AVAILABLE YET ({e!r}); installed in cell 2 if pinned")
import torch
print("torch.cuda.is_available:", torch.cuda.is_available(), "| visible devices:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> GPU T4 x2.")
gpu = torch.cuda.get_device_name(0)
print("GPU 0:", gpu)
if CONFIG["required_gpu_substring"] not in gpu:
    raise RuntimeError(f"GPU is {gpu!r}; this project requires a T4 (P100 has no fast INT8). "
                       "Settings -> Accelerator -> GPU T4 x2, then restart.")
if torch.cuda.device_count() != 1:
    raise RuntimeError(f"expected exactly 1 visible GPU after pinning, got {torch.cuda.device_count()}")

# ---- get the repo: private GitHub clone (Secret GH_TOKEN), else attached zip dataset
REPO = CONFIG["repo_dir"]
repo_source = None
if os.path.isdir(REPO):
    shutil.rmtree(REPO)
try:
    if "github.com/OWNER/" in CONFIG["repo_https"]:
        raise RuntimeError("repo_https is still the placeholder: set it in configs/kaggle.yaml and regenerate")
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    if not token:
        raise RuntimeError("Secret GH_TOKEN is empty")
    hdr = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    # token passed as a one-off header: never written to .git/config or printed
    r = subprocess.run(["git", "-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {hdr}",
                        "clone", "--quiet", CONFIG["repo_https"], REPO], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git clone failed: " + r.stderr.replace(token, "***").replace(hdr, "***"))
    sh(["git", "-C", REPO, "checkout", "--quiet", CONFIG["repo_ref"]])
    repo_source = "github"
    del token, hdr
except Exception as e:
    print(f"GitHub clone unavailable: {e!r}\nTrying zip-dataset fallback ...")
    hits = sorted(glob.glob("/kaggle/input/**/" + CONFIG["repo_zip_marker"], recursive=True))
    if not hits:
        raise RuntimeError("Neither GitHub clone (Secret GH_TOKEN) nor an attached repo zip "
                           "(dataset containing adapres/GIT_COMMIT) is available. See README cell.") from e
    shutil.copytree(os.path.dirname(hits[0]), REPO)
    repo_source = "zip_dataset:" + os.path.dirname(hits[0])

sys.path.insert(0, os.path.join(REPO, "src"))
sys.path.insert(0, os.path.join(REPO, "scripts"))
# Re-running cells in a live kernel re-clones the repo but Python would keep the OLD imported
# modules while GIT records the NEW commit. Drop them so the code always matches the hash.
_stale = [m for m in list(sys.modules) if m == "adapres" or m.startswith("adapres.") or m == "make_notebooks"]
for m in _stale:
    del sys.modules[m]
if _stale:
    print(f"dropped {len(_stale)} previously imported adapres modules (fresh code from this clone)")
import make_notebooks
expected_rev = make_notebooks.notebook_rev(CONFIG["notebook"] + ".ipynb")
if expected_rev != NOTEBOOK_REV:
    raise RuntimeError(f"STALE NOTEBOOK: this notebook is rev {NOTEBOOK_REV} but repo {CONFIG['repo_ref']} "
                       f"generates rev {expected_rev}. Re-import notebooks/{CONFIG['notebook']}.ipynb "
                       "from the repo (File -> Import Notebook), or push your local commits.")
print("notebook rev", NOTEBOOK_REV, "matches repo")
from adapres import config as C
from adapres.repro import git_commit, set_seed
cfg = C.load(os.path.join(REPO, CONFIG["config_path"]), CONFIG["overrides"])
SEED = cfg["seed"]
set_seed(SEED)
GIT = git_commit(REPO)
print("repo source:", repo_source)
print("git commit:", GIT)
print("seed:", SEED, "| SMOKE_TEST:", SMOKE_TEST)
if GIT["commit"] is None:
    raise RuntimeError("could not determine the git commit; refusing to produce unattributable results")
if GIT["dirty"]:
    raise RuntimeError("repo checkout is dirty; results would not match the recorded commit")
RESULTS = C.resolve_out(cfg, "results_dir", SMOKE_TEST)
LOGS = C.resolve_out(cfg, "logs_dir", SMOKE_TEST)
for d in (RESULTS, LOGS):
    os.makedirs(d, exist_ok=True)
print("results ->", RESULTS, "| logs ->", LOGS)


In [ ]:
# ===================== CELL 2: LOCKED INSTALLS + VERSION CHECK =====================
# Internet must be ON. Installs only what the Kaggle image lacks; torch is not touched.
# Every pin is exact (locked 2026-11-20); the cell fails if anything installs at another version.
import re, importlib.metadata as md
print("python", sys.version, "| torch", torch.__version__, "| torch CUDA", torch.version.cuda)
REQ = os.path.join(REPO, "requirements-kaggle.txt")
lines = [ln.split("#")[0].strip() for ln in open(REQ)]
opts = [tok for ln in lines if ln.startswith("-") for tok in ln.split()]  # e.g. --extra-index-url URL
print("pip options:", opts)
reqs = [ln for ln in lines if ln and not ln.startswith("-")]
PIP = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check"]
# Preflight: resolve each requirement on its own (no install) so ALL failures show in one run.
bad = {}
for req in reqs:
    p = subprocess.run(PIP + ["--dry-run", "--no-deps", "--quiet", *opts, req], capture_output=True, text=True)
    if p.returncode != 0:
        bad[req] = "\n".join(l for l in p.stderr.splitlines() if "ERROR" in l) or p.stderr[-2000:]
    print(f"preflight {req:45s} {'OK' if p.returncode == 0 else 'FAILED'}")
if bad:
    for req, err in bad.items():
        print(f"\n--- {req} ---\n{err}")
    raise RuntimeError(f"{len(bad)} requirement(s) have no wheel for this Python/CUDA: {list(bad)}. "
                       "Paste this whole cell output; the available versions are listed above.")
r = sh(PIP + ["--quiet", "-r", REQ], check=False)
with open(os.path.join(LOGS, f"{CONFIG['notebook']}_pip.log"), "w") as f:
    f.write(r.stdout + "\n--- stderr ---\n" + r.stderr)
if r.returncode != 0:
    raise RuntimeError("pip install failed (real error above).")
importlib.invalidate_caches()
# Version lock: record exactly what got installed so requirements-kaggle.txt can be pinned to it.
names = [re.split(r"[<>=!~\[ ]", q)[0] for q in reqs]
resolved = {n: md.version(n) for n in names}
os.makedirs(os.path.join(RESULTS, "env"), exist_ok=True)
lock = "\n".join(f"{n}=={v}" for n, v in resolved.items())
with open(os.path.join(RESULTS, "env", "pip_resolved.txt"), "w") as f:
    f.write(f"# python {sys.version.split()[0]} torch {torch.__version__} cuda {torch.version.cuda}\n{lock}\n")
print("RESOLVED VERSIONS (copy into requirements-kaggle.txt as exact pins):\n" + lock)
trt_dists = ("tensorrt-cu13", "tensorrt-cu13-bindings", "tensorrt-cu13-libs")
if len({resolved.get(d) for d in trt_dists}) != 1:
    raise RuntimeError(f"TensorRT wrapper/bindings/libs version mismatch: {resolved}")
pinned = {re.split(r"[<>=!~\[ ]", q)[0]: q.split("==")[-1] for q in reqs if "==" in q}
drift = {n: (pinned[n], resolved[n]) for n in pinned if pinned[n] != resolved[n]}
if drift:
    raise RuntimeError(f"installed versions differ from the lock (pinned, installed): {drift}")
try:
    import tensorrt
except ImportError:
    # Diagnose instead of guessing: which top-level modules did each TensorRT dist install?
    for d in trt_dists:
        try:
            tops = sorted({str(f).split("/")[0] for f in (md.files(d) or [])})
            print(f"{d}: top-level entries {tops[:25]}")
        except md.PackageNotFoundError:
            print(f"{d}: NOT INSTALLED")
    raise
import onnx, onnxruntime
print("import OK: tensorrt", tensorrt.__version__, "| onnx", onnx.__version__, "| onnxruntime", onnxruntime.__version__)
print("torch still", torch.__version__, "cuda", torch.version.cuda)


In [ ]:
# ===================== CELL 3: PREPARE (manifest, GT rebuild, weights) =====================
import logging
os.environ["YOLO_AUTOINSTALL"] = "False"   # never let Ultralytics pip-install unpinned packages
if CONFIG["model_override"]:
    cfg = C.load(os.path.join(REPO, CONFIG["config_path"]), {**CONFIG["overrides"], "model": CONFIG["model_override"]})
logging.basicConfig(level=logging.INFO, force=True, format="%(asctime)s %(levelname)s %(name)s: %(message)s",
                    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(os.path.join(LOGS, "02b_val_diag.log"))])
from adapres.pipeline.eval_fixed import Step2
WORK = os.path.join(cfg["paths"]["work_root"], "smoke" if SMOKE_TEST else "", "work02b")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)                              # Ultralytics downloads weights into the cwd
from ultralytics import YOLO
import ultralytics
print("ultralytics", ultralytics.__version__)
_m = YOLO(cfg["model"]["weights"])
WEIGHTS = os.path.abspath(cfg["model"]["weights"])
if not os.path.exists(WEIGHTS):
    raise RuntimeError(f"weights not found after download: {WEIGHTS}")
NAMES = dict(_m.names)
del _m
step = Step2(cfg, SMOKE_TEST, REPO)
step.coco_names = NAMES
t = time.time()
info = step.prepare()
print(json.dumps(info, indent=2)[:2000])
print(f"prepare: {time.time()-t:.0f}s | model {cfg['model']['name']} weights {WEIGHTS}")


In [ ]:
# ===================== CELL 4: A / B / C ON THE SAME 200 CALIB FRAMES =====================
from pathlib import Path
from adapres.models.engines import UltralyticsBackend
from adapres.pipeline import val_diag
t = time.time()
mc = cfg["model"]
pt = UltralyticsBackend("pt_fp32", WEIGHTS, mc["imgsz"], mc["conf"], mc["iou"], mc["max_det"], mc["rect"])
first = step.paths[step.image_ids[0]]
for _ in range(10):
    pt.predict(str(first))                      # warm-up, discarded

CAPTURED = {}
def _grab(validator):
    CAPTURED["jdict"] = list(getattr(validator, "jdict", []) or [])
    CAPTURED["save_dir"] = str(getattr(validator, "save_dir", ""))
    CAPTURED["args"] = {k: getattr(validator.args, k, None) for k in
                        ("conf", "iou", "max_det", "imgsz", "batch", "rect", "half", "save_json",
                         "agnostic_nms", "single_cls", "augment", "split")}

def val_fn(yaml_path):
    m = YOLO(WEIGHTS)
    m.add_callback("on_val_end", _grab)
    r = m.val(data=str(yaml_path), imgsz=mc["imgsz"], batch=1, conf=mc["conf"], iou=mc["iou"],
              max_det=mc["max_det"], rect=False, half=False, plots=False, save_json=True, verbose=False,
              device=0, project=os.path.join(WORK, "runs02b"), name="val", exist_ok=True)
    jd = CAPTURED.get("jdict")
    if not jd:  # fall back to the file Ultralytics writes after on_val_end
        pj = Path(CAPTURED.get("save_dir", "")) / "predictions.json"
        print("jdict empty in callback; reading", pj, "exists:", pj.exists())
        jd = json.loads(pj.read_text()) if pj.exists() else []
    return {"map50_95": float(r.box.map), "map50": float(r.box.map50), "jdict": jd, "args": CAPTURED.get("args")}

committed = os.path.join(REPO, "results", "sanity", f"sanity_02_{cfg['model']['name']}", "val_check.json")
committed = json.load(open(committed)) if os.path.exists(committed) else None
DIAG = val_diag.run(step, lambda p: pt.predict(p)[0], val_fn, cfg["step2"]["sanity"]["val_check_max_abs_pts"],
                    committed)
print("model.val() args actually used:", DIAG["ultralytics_val_args"])
print(f"diagnostic: {time.time()-t:.0f}s")


In [ ]:
# ===================== CELL 5: REPORT =====================
import pandas as pd
rows = []
for k, c in DIAG["components"].items():
    rows.append({"metric": k, "A ours/ours": c["A_ours_pred_ours_metric"], "B theirs/ours": c["B_their_pred_ours_metric"],
                 "C theirs/theirs": c["C_their_pred_their_metric"], "pred comp B-A": c["prediction_component_pts"],
                 "metric comp C-B": c["metric_component_pts"], "end-to-end C-A": c["end_to_end_pts"], "cause": c["cause"]})
pd.set_option("display.width", 250)
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nprediction similarity:", json.dumps(DIAG["prediction_similarity"], indent=1))
print("\nmulti-label NMS signature (same box, other class; images at max_det):")
for who, s in DIAG["multilabel_signature"].items():
    print(f"  {who:7s} boxes/img {s['mean_boxes_per_image']:.1f}  dup-other-class {s['frac_boxes_with_same_box_other_class']:.4f}"
          f"  at max_det {s['frac_images_at_max_det']:.3f}")
if "reproduces_notebook02" in DIAG:
    print("\nreproduction of notebook 02 numbers:", json.dumps(DIAG["reproduces_notebook02"], indent=1))
print("\nchecks (|diff| <= %.2f pts):" % DIAG["tolerance_pts"])
for name, ch in DIAG["checks"].items():
    print(f"  {name:22s} " + "  ".join(f"{k}: {'PASS' if v else 'FAIL'}" for k, v in ch.items()))
print(f"\nBINDING CHECK: {DIAG['binding_check']} -> {'PASS' if DIAG['binding_pass'] else 'FAIL'}")


In [ ]:
# ===================== FINAL CELL: ZIP RESULTS =====================
import zipfile
suffix = "_smoke" if SMOKE_TEST else ""
zpath = f"/kaggle/working/results_02b_val_diagnostic{suffix}.zip"
base = os.path.dirname(RESULTS)  # contains results/ and logs/
n = 0
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for top in (RESULTS, LOGS):
        for root, _, files in os.walk(top):
            for fn in files:
                full = os.path.join(root, fn)
                z.write(full, os.path.relpath(full, base))
                n += 1
print(f"wrote {zpath} ({n} files, {os.path.getsize(zpath)/1e6:.2f} MB)")
print(f"total notebook time: {(time.time()-T0)/60:.1f} min")
if not DIAG["binding_pass"]:
    raise RuntimeError(f"binding agreement check '{DIAG['binding_check']}' failed (zip written); see the report cell")
